# ChartQA — Qwen3-VL-8B quantization study (Kaggle T4)

Runs the **Phase 1.5 Stage B1 / C1** accuracy study for the roadmap: Qwen3-VL-8B at
**4-bit** (and optionally 8-bit), zero-shot vs LoRA fine-tuned, on the full 2500-sample
ChartQA test split. Accuracy is hardware-independent, so these T4 numbers equal what a
4050 would produce — the 4050 just can't *hold* the 8B at 4-bit (see roadmap B2).

**Reference (already measured, bf16):** zero-shot 84.60% / fine-tuned 86.08% relaxed.
Do **not** re-run bf16 — fill the 4-bit / 8-bit rows only.

## Prerequisites (do these first)
1. **Notebook settings → Accelerator = GPU T4 x1**, and **Internet = On** (for the HF download).
2. **Get the code onto Kaggle** — pick one:
   - *(recommended)* Upload the repo's `modeling/` folder as a **Kaggle Dataset**
     (include `checkpoints/qwen3vl-lora-final2` for the fine-tuned run — it's a small
     LoRA adapter). Then set `CODE_DIR` below to `/kaggle/input/<your-dataset>/modeling`.
   - *or* `git clone` your repo (needs the branch pushed + `git lfs` for the adapter).
3. *(optional)* Add your **HF token** via Kaggle **Add-ons → Secrets** as `HF_TOKEN`
   for faster, rate-limit-free downloads.

The base Qwen3-VL-8B (~17.5 GB) downloads from HF on first run into `/kaggle/working/hf`.

In [ ]:
# 1) Runtime deps. Kaggle already ships CUDA torch/torchvision, so install the rest
#    (do NOT reinstall torch — it can break the CUDA build).
!pip install -q -U transformers bitsandbytes accelerate peft datasets "mlflow>=2.14,<3"
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("transformers", transformers.__version__)  # needs >=4.57 for Qwen3-VL

In [ ]:
# 2) Point Python at the chartqa package (no install needed — read-only import).
#    EDIT CODE_DIR to where you uploaded the `modeling/` folder.
import os, sys

CODE_DIR = "/kaggle/input/chartqa-modeling/modeling"   # <-- EDIT ME (dataset path)
# Alternative — git clone (uncomment; needs a pushed branch + git-lfs for the adapter):
# !git lfs install
# !git clone --branch feat/quantization-flag --depth 1 https://github.com/<you>/Chart-Visual-QA.git /kaggle/working/repo
# CODE_DIR = "/kaggle/working/repo/modeling"

assert os.path.isdir(os.path.join(CODE_DIR, "chartqa")), f"chartqa/ not found under {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
# ALSO set PYTHONPATH: `!python -m ...` (cell 4) and subprocess.run (cell 6) spawn a
# separate process that does NOT inherit sys.path — only os.environ. Without this,
# those cells fail with "ModuleNotFoundError: No module named 'chartqa'" even though
# `import chartqa` works fine right here in the kernel.
os.environ["PYTHONPATH"] = CODE_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

# LoRA adapter for the fine-tuned runs (set to None to skip fine-tuned configs).
ADAPTER = os.path.join(CODE_DIR, "checkpoints", "qwen3vl-lora-final2")
ADAPTER = ADAPTER if os.path.isdir(ADAPTER) else None
print("code:", CODE_DIR, "| adapter:", ADAPTER)
print("PYTHONPATH:", os.environ["PYTHONPATH"])

In [ ]:
# 3) Caches + tracking. HF cache goes to /tmp (scratch, big, NOT counted against the
#    ~20 GB /kaggle/working output quota — the base model alone is ~17.5 GB). MLflow
#    runs are tiny, so they live in /kaggle/working so you can download them.
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["MLFLOW_TRACKING_URI"] = "file:/kaggle/working/mlruns"
os.environ["MLFLOW_EXPERIMENT"] = "chartqa-eval"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("No HF_TOKEN secret (downloads still work, just rate-limited):", e)

os.chdir("/kaggle/working")  # eval writes outputs/ here

In [ ]:
# 4) SMOKE TEST (roadmap Stage A2) — 50 samples, zero-shot 4-bit. Confirms the real path
#    (Qwen + bitsandbytes 4-bit + metric + MLflow) works BEFORE the long full runs.
#    First run also downloads the base model (~17.5 GB) — expect several minutes.
!python -m chartqa.evaluation.evaluate --model qwen --quantization 4bit --limit 50 --metric relaxed

## Full runs (Stage B1 = 4-bit; Stage C1 = 8-bit)

Each config is a full 2500-sample generation pass (~1-2 h on a T4). `relaxed` and `exact`
are separate passes in the current harness, so a full 4-config × 2-metric sweep is long —
**run `relaxed` first** (the headline metric); do `exact` only if time allows. Everything
logs to MLflow (peak VRAM, latency, load time, accuracy). Comment out the 8-bit rows to
stay inside one session.

In [ ]:
# 5) Full 2500-sample runs. (model, adapter-or-None, quantization, metric)
import subprocess

CONFIGS = [("qwen", None, "4bit", "relaxed")]            # B1: zero-shot 4-bit (always)
if ADAPTER:
    CONFIGS.append(("qwen", ADAPTER, "4bit", "relaxed"))  # B1: fine-tuned 4-bit
# Optional extras (each is another full ~2500-sample pass — uncomment as time allows):
# CONFIGS += [("qwen", None, "8bit", "relaxed")]           # C1: zero-shot 8-bit (~10 GB)
# if ADAPTER: CONFIGS.append(("qwen", ADAPTER, "8bit", "relaxed"))
# CONFIGS += [("qwen", None, "4bit", "exact")]             # exact-match second pass

for model, adapter, quant, metric in CONFIGS:
    cmd = ["python", "-m", "chartqa.evaluation.evaluate",
           "--model", model, "--quantization", quant, "--metric", metric]
    if adapter:
        cmd += ["--checkpoint", adapter]
    print("\n>>>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
# 6) Roll-up: read the tracked runs into a table (the roadmap comparison table).
import mlflow, pandas as pd
pd.set_option("display.width", 200)
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
df = mlflow.search_runs(experiment_names=["chartqa-eval"])
cols = [c for c in [
    "tags.mlflow.runName", "params.checkpoint", "params.quantization", "params.metric",
    "metrics.relaxed_accuracy", "metrics.exact_accuracy", "metrics.peak_vram_gb",
    "metrics.latency_p50_s", "metrics.latency_p95_s", "metrics.load_time_s",
] if c in df.columns]
display(df[cols].sort_values(cols[0]))
# Download everything: zip the tracking dir, then grab it from the Output tab.
!cd /kaggle/working && zip -qr mlruns.zip mlruns && ls -lh mlruns.zip